# SageMaker Pipelines with MLflow

https://github.com/aws/amazon-sagemaker-examples/blob/main/sagemaker-mlflow/sagemaker_pipelines_mlflow.ipynb

## Run in root directory rather than shared/ for reduced s3 spend and better performance

## Setup environment

Import necessary libraries

In [1]:
import os

import sagemaker
from sagemaker.workflow.execution_variables import ExecutionVariables
from sagemaker.workflow.function_step import step
from sagemaker.workflow.parameters import ParameterString
from sagemaker.workflow.pipeline import Pipeline
from sagemaker.workflow.condition_step import ConditionStep
from sagemaker.workflow.conditions import ConditionGreaterThanOrEqualTo
from sagemaker.workflow.fail_step import FailStep

sagemaker.config INFO - Fetched defaults config from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/sagemaker-user/.config/sagemaker/config.yaml
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix


Declare some variables used later

### Change these constants:

In [2]:
LABEL_COLUMN = 'tlu_loss'
FEAT_LS = ['soil', 'ppt', 'pdsi', 'vpd', 'ndvi', 'lai', 'lst', 'soil_lag1', 'soil_lag2', 'soil_lag3',
                     'ppt_lag1', 'ppt_lag2', 'ppt_lag3', 'pdsi_lag1', 'pdsi_lag2', 'pdsi_lag3', 'vpd_lag1',
                     'vpd_lag2', 'vpd_lag3', 'ndvi_lag1', 'ndvi_lag2', 'ndvi_lag3', 'lai_lag1', 'lai_lag2',
                     'lai_lag3', 'lst_lag1', 'lst_lag2', 'lst_lag3', 'month_sin', 'month_cos',]

# XGB parameters
use_gpu = False
param = dict(
    objective="reg:squarederror",
    max_depth=5,
    eta=0.2,
    gamma=4,
    min_child_weight=6,
    subsample=0.7,
    tree_method="gpu_hist" if use_gpu else "hist",  # Use GPU accelerated algorithm
    early_stopping_rounds=10 # stop training if val loss not improving every x rounds
)
num_round = 50

In [3]:
sagemaker_session = sagemaker.session.Session()
role = sagemaker.get_execution_role()
bucket = sagemaker_session.default_bucket()
region = sagemaker_session.boto_region_name

pipeline_name = "lmr-xgb-training-pipeline"
instance_type = ParameterString(name="TrainingInstanceType", default_value="ml.m5.xlarge")

# Mlflow
tracking_server_arn = "arn:aws:sagemaker:us-east-1:575108933641:mlflow-tracking-server/lmr-tracking-server-5t7l23o0xvt99j-chws71x3trpelj-dev"
experiment_name = "lmr-sm-pipelines-experiment"

sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix


Write `requirements` and `config` files that'll be used by the steps in our SageMaker Pipeline

In [4]:
%%writefile config.yaml
SchemaVersion: '1.0'
SageMaker:
  PythonSDK:
    Modules:
      RemoteFunction:
        # role arn is not required if in SageMaker Notebook instance or SageMaker Studio
        # Uncomment the following line and replace with the right execution role if in a local IDE
        # RoleArn: <replace the role arn here>
        InstanceType: ml.m5.xlarge
        Dependencies: ./requirements.txt
        IncludeLocalWorkDir: true
        CustomFileFilter:
          IgnoreNamePatterns: # files or directories to ignore
          - "*.ipynb" # all notebook files

Overwriting config.yaml


In [5]:
# Set path to config file
os.environ["SAGEMAKER_USER_CONFIG_OVERRIDE"] = os.getcwd()

In [6]:
%%writefile requirements.txt
scikit-learn
xgboost==2.1.4
s3fs==2024.12.0
sagemaker==2.254.1
pandas>=2.0.0
gevent
geventhttpclient
shap
matplotlib
fsspec
mlflow==2.13.2
sagemaker-mlflow==0.1.0
cloudpickle==3.1.2

Overwriting requirements.txt


## Define the SageMaker Pipeline

### Preprocessing Step

In [7]:
# Location of our dataset
file_name = "monthly_pred_2008_2016.parquet"
input_path = f"s3://amazon-sagemaker-575108933641-us-east-1-c422b90ce861/dzd-ayr06tncl712p3/5t7l23o0xvt99j/dev/data/training/processed/{file_name}"

The breast cancer Wisconsin dataset contains column `id` which we do not use for training. The second column `diagnosis` is class label, and the label is represented using 'M' for Malignant class, and 'B' for Benign class. 

In the preprocessing step, we drop the column `id`, then split the dataset into three distinct sets: train, validation, and test set.

Note that `keep_alive_period_in_seconds` parameter in @step decorator indicates how many seconds we want to keep the instance alive, waiting to be reused for the next pipeline step execution. Setting this parameter speeds up the pipeline execution because we reduce the launching of new instances to execute pipeline steps.

In [8]:
# TODO: update preprocessing to match Skylar's notebook
@step(
    name="DataPreprocessing",
    instance_type=instance_type,
)
def preprocess(raw_data_s3_path, output_prefix, experiment_name, run_name, label_column, test_size, feature_names):
    from preprocess import run_preprocess  # imported at runtime on the remote instance
    # returns train_s3_path, val_s3_path, test_s3_path, experiment_name, run_id
    return run_preprocess(raw_data_s3_path, output_prefix, experiment_name, run_name, label_column, test_size, feature_names)

### Training Step

We train a XGBoost model in this training step, using @step-decorated function with the S3 path of training and validation set, along with XGBoost hyperparameters. The S3 paths for both training and validation set is coming from the output of the previous step.

This could be replaced or supplemented with an option for selecting a different model architecture

In [9]:
@step(
    name="ModelTraining",
    instance_type=instance_type,
)
def train(
    train_s3_path: str,
    validation_s3_path: str,
    experiment_name: str,
    run_id: str,
    param: dict = param,
    num_round: int = num_round,
    label_column: str = 'tlu_loss'
):
    import mlflow
    import pandas as pd
    from xgboost import XGBClassifier, XGBRegressor

    mlflow.set_tracking_uri(tracking_server_arn)
    mlflow.set_experiment(experiment_name)

    with mlflow.start_run(run_id=run_id):
        mlflow.xgboost.autolog(
                log_input_examples=True,
                log_model_signatures=True,
                log_models=False,
                log_datasets=True,
                model_format="xgb",
            )
        with mlflow.start_run(run_name="ModelTraining", nested=True) as training_run:
            training_run_id = training_run.info.run_id

            # read data files from S3
            train_df = pd.read_csv(train_s3_path)
            validation_df = pd.read_csv(validation_s3_path)

            # create dataframe and label series
            y_train = train_df.pop(label_column).astype("float")
            y_validation = validation_df.pop(label_column).astype("float")
            xgb = XGBRegressor(n_estimators=num_round, **param)
            xgb.fit(
                train_df,
                y_train,
                eval_set=[(validation_df, y_validation)],
                # early_stopping_rounds=5,
            )
            
            # Explicitly log model to the nested run so we know exactly where it is
            mlflow.sklearn.log_model(
                xgb,
                artifact_path="model",
                input_example=train_df[:5],
            )

        # return xgb
        return experiment_name, run_id, training_run_id

### Evaluation Step

In this step, we create a @step-decorated function to evaluate the trained XGBoost model on the test dataset.

In [10]:
# @step(
#     name="ModelEvaluation",
#     instance_type=instance_type, 
# )
# def evaluate(
#     test_s3_path: str,
#     experiment_name: str,
#     run_id: str,
#     training_run_id: str,
# ) -> dict:
#     import mlflow
#     import pandas as pd

#     mlflow.set_tracking_uri(tracking_server_arn)
#     mlflow.set_experiment(experiment_name)

#     with mlflow.start_run(run_id=run_id):
#         with mlflow.start_run(run_name="ModelEvaluation", nested=True):
#             test_df = pd.read_csv(test_s3_path)
#             # test_df[label_column] = (test_df[label_column] == "M").astype("int")
#             # Removing target processing step from example binary classification task
#             model = mlflow.pyfunc.load_model(f"runs:/{training_run_id}/model")

#             results = mlflow.evaluate(
#                 model=model,
#                 data=test_df,
#                 targets=label_column,
#                 model_type="regressor", # "classifier"
#                 evaluators=["default"],
#             )
#             return {"f1_score": results.metrics["f1_score"]}

In [11]:
@step(name="ModelEvaluation", instance_type=instance_type)
def evaluate(test_s3_path, experiment_name, run_id, training_run_id, label_column='tlu_loss') -> dict:
    import mlflow
    import pandas as pd
    import numpy as np
    from sklearn.metrics import (
        mean_squared_error,
        mean_absolute_error,
        mean_absolute_percentage_error,
        r2_score,
    )

    mlflow.set_tracking_uri(tracking_server_arn)
    mlflow.set_experiment(experiment_name)

    with mlflow.start_run(run_id=run_id):
        with mlflow.start_run(run_name="ModelEvaluation", nested=True):

            test_df = pd.read_csv(test_s3_path)
            y_true = test_df.pop(label_column).astype(float)
            X_test = test_df

            # Debug: confirm data loaded correctly
            print(f"Test set shape: {X_test.shape}")
            print(f"y_true sample: {y_true[:5].tolist()}")
            print(f"y_true range: {y_true.min():.3f} to {y_true.max():.3f}")

            model = mlflow.pyfunc.load_model(f"runs:/{training_run_id}/model")
            y_pred = model.predict(X_test)
            y_pred = np.array(y_pred).flatten().astype(float)

            print(f"y_pred sample: {y_pred[:5].tolist()}")
            print(f"y_pred range: {y_pred.min():.3f} to {y_pred.max():.3f}")

            # Core metrics
            rmse = np.sqrt(mean_squared_error(y_true, y_pred))
            mae  = mean_absolute_error(y_true, y_pred)
            r2   = r2_score(y_true, y_pred)

            # Normalized metrics (useful when label scale varies)
            label_range = y_true.max() - y_true.min()
            nrmse = rmse / label_range if label_range > 0 else float("nan")  # 0–1 scale
            mape  = mean_absolute_percentage_error(y_true, y_pred)           # % error

            # Naive baseline comparison
            naive_pred  = np.full_like(y_true, y_true.mean())
            naive_rmse  = np.sqrt(mean_squared_error(y_true, naive_pred))
            skill_score = 1 - (rmse / naive_rmse)  # >0 means better than naive mean

            metrics = {
                "rmse":        rmse,
                "mae":         mae,
                "r2_score":    r2,
                "nrmse":       nrmse,       # normalized RMSE
                "mape":        mape,        # mean absolute % error
                "naive_rmse":  naive_rmse,  # baseline to beat
                "skill_score": skill_score, # how much better than naive (>0 is good)
            }

            print(f"Metrics: {metrics}")
            mlflow.log_metrics(metrics)  # log all at once, inside the nested run

            return metrics

### Model Registration

In this step, we create a @step-decorated function to register our XGBoost model. 

In [12]:
@step(
    name="ModelRegistration",
    instance_type=instance_type,
)
def register(
    pipeline_name: str,
    experiment_name: str,
    run_id: str,
    training_run_id: str,
):
    import mlflow

    mlflow.set_tracking_uri(tracking_server_arn)
    mlflow.set_experiment(experiment_name)

    with mlflow.start_run(run_id=run_id):
        with mlflow.start_run(run_name="ModelRegistration", nested=True):
            mlflow.register_model(f"runs:/{training_run_id}/model", pipeline_name)

## Creating the SageMaker Pipeline

We connect all defined pipeline `@step` functions into a multi-step pipeline. Then, we submit and execute the pipeline.

In [13]:
preprocessing_step = preprocess(
    raw_data_s3_path=input_path,
    output_prefix=f"{pipeline_name}/dataset",
    experiment_name=experiment_name,
    run_name=ExecutionVariables.PIPELINE_EXECUTION_ID,
    label_column=LABEL_COLUMN,
    test_size=0.2, # 80 train/10 val/10 test
    feature_names=FEAT_LS
)
# returns train_s3_path, val_s3_path, test_s3_path, experiment_name, run_id

training_step = train(
    train_s3_path=preprocessing_step[0],
    validation_s3_path=preprocessing_step[1],
    experiment_name=preprocessing_step[3],
    run_id=preprocessing_step[4],
    label_column=LABEL_COLUMN,
)

# TODO: Update condition to be appropriate for regression task
conditional_register_step = ConditionStep(
    name="ConditionalRegister",
    conditions=[
        ConditionGreaterThanOrEqualTo(
            left=evaluate(
                test_s3_path=preprocessing_step[2],
                experiment_name=preprocessing_step[3],
                run_id=preprocessing_step[4],
                training_run_id=training_step[2],
                label_column=LABEL_COLUMN
            )["r2_score"],
            right=0, # only log if r2 is >= 0
        )
    ],
    if_steps=[
        register(
            pipeline_name=pipeline_name,
            experiment_name=preprocessing_step[3],
            run_id=preprocessing_step[4],
            training_run_id=training_step[2],
        )
    ],
    else_steps=[FailStep(name="Fail", error_message="Model performance is not good enough")],
)

pipeline = Pipeline(
    name=pipeline_name,
    parameters=[
        instance_type,
    ],
    steps=[preprocessing_step, training_step, conditional_register_step],
)

sagemaker.config INFO - Fetched defaults config from location: /home/sagemaker-user
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3Bucket
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.Session.DefaultS3ObjectKeyPrefix


In [14]:
pipeline.upsert(role_arn=role)

sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.RemoteFunction.Dependencies
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.RemoteFunction.IncludeLocalWorkDir
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.RemoteFunction.CustomFileFilter.IgnoreNamePatterns
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.RemoteFunction.VpcConfig.Subnets
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.RemoteFunction.VpcConfig.SecurityGroupIds


2026-03-08 22:38:08,274 sagemaker.remote_function INFO     Uploading serialized function code to s3://amazon-sagemaker-575108933641-us-east-1-c422b90ce861/dzd-ayr06tncl712p3/5t7l23o0xvt99j/dev/lmr-xgb-training-pipeline/DataPreprocessing/2026-03-08-22-38-05-975/function
2026-03-08 22:38:08,503 sagemaker.remote_function INFO     Uploading serialized function arguments to s3://amazon-sagemaker-575108933641-us-east-1-c422b90ce861/dzd-ayr06tncl712p3/5t7l23o0xvt99j/dev/lmr-xgb-training-pipeline/DataPreprocessing/2026-03-08-22-38-05-975/arguments
2026-03-08 22:38:09,189 sagemaker.remote_function INFO     Copied dependencies file at './requirements.txt' to '/tmp/tmp2dq9rkhh/requirements.txt'
2026-03-08 22:38:09,336 sagemaker.remote_function INFO     Successfully uploaded dependencies and pre execution scripts to 's3://amazon-sagemaker-575108933641-us-east-1-c422b90ce861/dzd-ayr06tncl712p3/5t7l23o0xvt99j/dev/lmr-xgb-training-pipeline/DataPreprocessing/2026-03-08-22-38-05-975/pre_exec_script_and

sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.RemoteFunction.Dependencies
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.RemoteFunction.IncludeLocalWorkDir
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.RemoteFunction.CustomFileFilter.IgnoreNamePatterns
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.RemoteFunction.VpcConfig.Subnets
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.RemoteFunction.VpcConfig.SecurityGroupIds


2026-03-08 22:38:19,731 sagemaker.remote_function INFO     Uploading serialized function code to s3://amazon-sagemaker-575108933641-us-east-1-c422b90ce861/dzd-ayr06tncl712p3/5t7l23o0xvt99j/dev/lmr-xgb-training-pipeline/ModelTraining/2026-03-08-22-38-05-975/function
2026-03-08 22:38:20,045 sagemaker.remote_function INFO     Uploading serialized function arguments to s3://amazon-sagemaker-575108933641-us-east-1-c422b90ce861/dzd-ayr06tncl712p3/5t7l23o0xvt99j/dev/lmr-xgb-training-pipeline/ModelTraining/2026-03-08-22-38-05-975/arguments
2026-03-08 22:38:20,359 sagemaker.remote_function INFO     Copied dependencies file at './requirements.txt' to '/tmp/tmpk_uad9lr/requirements.txt'
2026-03-08 22:38:20,526 sagemaker.remote_function INFO     Successfully uploaded dependencies and pre execution scripts to 's3://amazon-sagemaker-575108933641-us-east-1-c422b90ce861/dzd-ayr06tncl712p3/5t7l23o0xvt99j/dev/lmr-xgb-training-pipeline/ModelTraining/2026-03-08-22-38-05-975/pre_exec_script_and_dependencie

sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.RemoteFunction.Dependencies
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.RemoteFunction.IncludeLocalWorkDir
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.RemoteFunction.CustomFileFilter.IgnoreNamePatterns
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.RemoteFunction.VpcConfig.Subnets
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.RemoteFunction.VpcConfig.SecurityGroupIds


2026-03-08 22:38:22,545 sagemaker.remote_function INFO     Uploading serialized function code to s3://amazon-sagemaker-575108933641-us-east-1-c422b90ce861/dzd-ayr06tncl712p3/5t7l23o0xvt99j/dev/lmr-xgb-training-pipeline/ModelRegistration/2026-03-08-22-38-05-975/function
2026-03-08 22:38:22,852 sagemaker.remote_function INFO     Uploading serialized function arguments to s3://amazon-sagemaker-575108933641-us-east-1-c422b90ce861/dzd-ayr06tncl712p3/5t7l23o0xvt99j/dev/lmr-xgb-training-pipeline/ModelRegistration/2026-03-08-22-38-05-975/arguments
2026-03-08 22:38:23,142 sagemaker.remote_function INFO     Copied dependencies file at './requirements.txt' to '/tmp/tmpftlgdeyo/requirements.txt'
2026-03-08 22:38:23,309 sagemaker.remote_function INFO     Successfully uploaded dependencies and pre execution scripts to 's3://amazon-sagemaker-575108933641-us-east-1-c422b90ce861/dzd-ayr06tncl712p3/5t7l23o0xvt99j/dev/lmr-xgb-training-pipeline/ModelRegistration/2026-03-08-22-38-05-975/pre_exec_script_and

sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.RemoteFunction.Dependencies
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.RemoteFunction.IncludeLocalWorkDir
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.RemoteFunction.CustomFileFilter.IgnoreNamePatterns
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.RemoteFunction.VpcConfig.Subnets
sagemaker.config INFO - Applied value from config key = SageMaker.PythonSDK.Modules.RemoteFunction.VpcConfig.SecurityGroupIds


2026-03-08 22:38:25,351 sagemaker.remote_function INFO     Uploading serialized function code to s3://amazon-sagemaker-575108933641-us-east-1-c422b90ce861/dzd-ayr06tncl712p3/5t7l23o0xvt99j/dev/lmr-xgb-training-pipeline/ModelEvaluation/2026-03-08-22-38-05-975/function
2026-03-08 22:38:25,653 sagemaker.remote_function INFO     Uploading serialized function arguments to s3://amazon-sagemaker-575108933641-us-east-1-c422b90ce861/dzd-ayr06tncl712p3/5t7l23o0xvt99j/dev/lmr-xgb-training-pipeline/ModelEvaluation/2026-03-08-22-38-05-975/arguments
2026-03-08 22:38:25,993 sagemaker.remote_function INFO     Copied dependencies file at './requirements.txt' to '/tmp/tmpy_jfm0vh/requirements.txt'
2026-03-08 22:38:26,146 sagemaker.remote_function INFO     Successfully uploaded dependencies and pre execution scripts to 's3://amazon-sagemaker-575108933641-us-east-1-c422b90ce861/dzd-ayr06tncl712p3/5t7l23o0xvt99j/dev/lmr-xgb-training-pipeline/ModelEvaluation/2026-03-08-22-38-05-975/pre_exec_script_and_depen

{'PipelineArn': 'arn:aws:sagemaker:us-east-1:575108933641:pipeline/lmr-xgb-training-pipeline',
 'PipelineVersionId': 14,
 'ResponseMetadata': {'RequestId': 'f5405dc9-08fc-47c7-98f7-d01a0268f885',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'x-amzn-requestid': 'f5405dc9-08fc-47c7-98f7-d01a0268f885',
   'strict-transport-security': 'max-age=47304000; includeSubDomains',
   'x-frame-options': 'DENY',
   'content-security-policy': "frame-ancestors 'none'",
   'cache-control': 'no-cache, no-store, must-revalidate',
   'x-content-type-options': 'nosniff',
   'content-type': 'application/x-amz-json-1.1',
   'content-length': '116',
   'date': 'Sun, 08 Mar 2026 22:38:37 GMT'},
  'RetryAttempts': 0}}

## Execute the SageMaker Pipeline

In [15]:
execution = pipeline.start()

In [16]:
execution.describe()

{'PipelineArn': 'arn:aws:sagemaker:us-east-1:575108933641:pipeline/lmr-xgb-training-pipeline',
 'PipelineExecutionArn': 'arn:aws:sagemaker:us-east-1:575108933641:pipeline/lmr-xgb-training-pipeline/execution/igh6oibbl5cr',
 'PipelineExecutionDisplayName': 'execution-1773009517928',
 'PipelineExecutionStatus': 'Executing',
 'CreationTime': datetime.datetime(2026, 3, 8, 22, 38, 37, 866000, tzinfo=tzlocal()),
 'LastModifiedTime': datetime.datetime(2026, 3, 8, 22, 38, 37, 866000, tzinfo=tzlocal()),
 'CreatedBy': {'UserProfileArn': 'arn:aws:sagemaker:us-east-1:575108933641:user-profile/d-lf2eivtg2b7i/c4083418-d061-7068-829e-604008d36196',
  'UserProfileName': 'c4083418-d061-7068-829e-604008d36196',
  'DomainId': 'd-lf2eivtg2b7i',
  'IamIdentity': {'Arn': 'arn:aws:sts::575108933641:assumed-role/datazone_usr_role_5t7l23o0xvt99j_5w5uuggwdfhu3r/SageMaker',
   'PrincipalId': 'AROAYLZZJ5AEZSW7N2N4C:SageMaker',
   'SourceIdentity': 'c4083418-d061-7068-829e-604008d36196'}},
 'LastModifiedBy': {'User

In [ ]:
execution.wait()

In [ ]:
execution.list_steps()